# Reproduksi CodeQA — CodeBERT (EMNLP 2021)

Notebook ini mengulang **baseline CodeBERT**, memakai dataset, arsitektur, preprocessing, beam search, dan evaluator dari penulis.
Bukan reproduksi keempat baseline atau pembangunan ulang dataset dari komentar.

**Cara pakai:** buka di Jupyter atau Google Colab, pilih GPU, lalu Run All. Default `MODE="smoke"` hanya menguji alur.
Ganti `MODE="full"` untuk fine-tuning 20 epoch. Untuk menguji checkpoint penulis tanpa training, gunakan `MODE="evaluate"`.
Skor smoke dapat bernilai nol karena decoder baru hanya dilatih pada delapan contoh.
Jalankan kembali dari kernel bersih jika mengganti bahasa atau mode.

Sumber primer:
- [Paper](https://aclanthology.org/2021.findings-emnlp.223/) — Liu & Wan, 2021.
- [Source code resmi](https://github.com/jadecxliu/CodeQA), commit `e0dba3ccb260447096e20ad1704013272512c779`.
- [Dataset resmi v1.0](https://drive.google.com/drive/folders/1i04sJNUHwMuDfMV2UfWeQG-Uv8MRw_qh).
- [Checkpoint QA penulis](https://drive.google.com/drive/folders/1A_C6O649cXjjpk3KKHIe6eaEU5tBaMLJ).
- [Pretrained encoder Microsoft CodeBERT](https://huggingface.co/microsoft/codebert-base).

Checkpoint QA berbeda dari pretrained encoder: encoder saja belum bisa menjawab pertanyaan.
Dataset dan kode yang diunduh diperlakukan sebagai data referensi; potongan kode dataset tidak dieksekusi.

## 1. Dependensi

Python 3.10+ dan PyTorch 2.6+ diperlukan. Notebook mempertahankan PyTorch bawaan Colab agar cocok dengan CUDA.
Jika PyTorch belum tersedia, instal melalui [petunjuk resmi PyTorch](https://pytorch.org/get-started/locally/) sebelum menjalankan notebook.
METEOR memakai Java dan resource penulis, bukan pengganti NLTK. Instal Java otomatis hanya pada Colab.

In [ ]:
import sys, subprocess, importlib.util, shutil
assert importlib.util.find_spec("torch"), "Instal PyTorch untuk CPU/CUDA sebelum melanjutkan."
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.57.1", "gdown==5.2.0", "nltk==3.9.2",
                "psutil==7.2.2", "tqdm==4.70.1"], check=True)
if shutil.which("java") is None and importlib.util.find_spec("google.colab"):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "default-jre-headless"], check=True)
assert shutil.which("java"), "METEOR membutuhkan Java: instal JRE, kemudian jalankan ulang."
assert shutil.which("git"), "Instal Git terlebih dahulu."
print("Dependensi siap; PyTorch dan Java tersedia.")

## 2. Konfigurasi dan batas reproduksi

| Parameter | Paper Appendix B | Skrip resmi | Notebook full |
|---|---|---|---|
| Epoch | 20 | 20 | 20 |
| Batch training | 64 | 32 | pilihan `BATCH_REFERENCE` |
| Learning rate | 5e-5 | 5e-5 | 5e-5 |
| Beam | 10 | 10 | 10 |
| Maksimum input | tidak dirinci | Java 200 / Python 400 | mengikuti skrip |
| Maksimum target | tidak dirinci | 30 | 30 |
| Decoder | 6 layer, 12 head | sama | sama |

Default full mengikuti batch **paper**. Batch dipecah melalui gradient accumulation untuk mengurangi VRAM;
ini tidak menjamin hasil numerik identik dengan training multi-GPU penulis.
Arsitektur mempertahankan default decoder skrip resmi, termasuk feedforward 2048 dan dropout 0.1.
Scheduler mengikuti kode resmi: warmup 10% lalu linear decay; weight decay 0.
Tidak ada mixed precision atau pemendekan input dalam mode full.

Perbaikan kompatibilitas dicatat: alokasi beam mengikuti device, import BLEU lama diperbaiki,
opsi JVM METEOR diurutkan ulang, dan `AdamW` lama diganti `torch.optim.AdamW`.
Buffer indeks posisi pada checkpoint lama diverifikasi lalu dihapus jika tidak lagi disimpan Transformers modern.
Loop training baru menangani batas accumulation dengan benar; loop upstream memiliki off-by-one saat accumulation >1.

In [ ]:
from pathlib import Path
import os, json, random, math, hashlib, platform, time, copy, gc
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
import numpy as np
import torch
assert tuple(map(int, torch.__version__.split("+")[0].split(".")[:2])) >= (2, 6), "Gunakan PyTorch 2.6+ untuk pemuatan bobot encoder."

LANGUAGE = "python"                # "python" atau "java"
MODE = "smoke"                     # "smoke", "full", atau "evaluate"
BATCH_REFERENCE = "paper"          # "paper" (64) atau "official_script" (32)
MICRO_BATCH_SIZE = 1                # naikkan jika VRAM cukup
EVAL_BATCH_SIZE = 1
CHECKPOINT_PATH = None              # evaluate: None mengunduh checkpoint QA resmi
SEED = 42

assert LANGUAGE in {"python", "java"}
assert MODE in {"smoke", "full", "evaluate"}
assert BATCH_REFERENCE in {"paper", "official_script"}
WORK = Path.cwd()
WORK.mkdir(exist_ok=True)
REPO = WORK / "source"
DATA = WORK / "data" / LANGUAGE
SOURCE_COMMIT = "e0dba3ccb260447096e20ad1704013272512c779"
MODEL_ID = "microsoft/codebert-base"
MODEL_REVISION = "3b0952feddeffad0063f274080e3c23d75e7eb39"
MAX_SOURCE_LENGTH = {"java": 200, "python": 400}[LANGUAGE]
MAX_TARGET_LENGTH = 30
EPOCHS = 1 if MODE == "smoke" else 20
BEAM_SIZE = 2 if MODE == "smoke" else 10
EFFECTIVE_BATCH_SIZE = 2 if MODE == "smoke" else (64 if BATCH_REFERENCE == "paper" else 32)
assert 0 < MICRO_BATCH_SIZE <= EFFECTIVE_BATCH_SIZE
assert EFFECTIVE_BATCH_SIZE % MICRO_BATCH_SIZE == 0
ACCUMULATION = EFFECTIVE_BATCH_SIZE // MICRO_BATCH_SIZE
LIMITS = {"train": 8, "dev": 4, "test": 4} if MODE == "smoke" else {}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if MODE == "smoke" and DEVICE.type == "cuda" and torch.cuda.get_device_properties(0).total_memory < 6 * 2**30:
    DEVICE = torch.device("cpu")
    print("VRAM di bawah 6 GiB: smoke memakai CPU agar dapat selesai tanpa memendekkan input.")
RUN = WORK / "runs" / f"{LANGUAGE}_{MODE}_{time.strftime('%Y%m%d_%H%M%S')}"
RUN.mkdir(parents=True, exist_ok=False)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.set_num_threads(min(4, os.cpu_count() or 1))
print({"language": LANGUAGE, "mode": MODE, "device": str(DEVICE),
       "epochs": EPOCHS, "effective_batch": EFFECTIVE_BATCH_SIZE,
       "micro_batch": MICRO_BATCH_SIZE, "beam": BEAM_SIZE})
if DEVICE.type == "cuda":
    print(torch.cuda.get_device_name(), round(torch.cuda.get_device_properties(0).total_memory/2**30, 1), "GiB")
print("Uji smoke bukan hasil reproduksi paper." if MODE == "smoke" else "Evaluasi akhir memakai seluruh test split.")

## 3. Ambil source code dengan versi tetap

Checkout upstream tidak diubah. Salinan kerja kompatibilitas dibuat terpisah dan perubahan disimpan sebagai diff.
Berkas upstream `run.py` dipakai untuk pembacaan data, tokenisasi, dan EM/F1; fungsi `main()` upstream tidak dijalankan.

In [ ]:
import difflib
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/jadecxliu/CodeQA.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", SOURCE_COMMIT], check=True)
actual_commit = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
if actual_commit != SOURCE_COMMIT:
    raise RuntimeError(f"Checkout harus pada {SOURCE_COMMIT}; gunakan folder kerja baru. Ditemukan {actual_commit}")
assert not subprocess.check_output(["git", "-C", str(REPO), "diff", "HEAD", "--"], text=True), "Source upstream berubah."
COMPAT = RUN / "compat"
shutil.copytree(REPO / "codeBERT" / "code", COMPAT, ignore=shutil.ignore_patterns("__pycache__"))

def replace_once(path, old, new):
    source = path.read_text()
    assert source.count(old) == 1, (path.name, old)
    updated = source.replace(old, new)
    path.write_text(updated)
    return "".join(difflib.unified_diff(source.splitlines(True), updated.splitlines(True),
                                        fromfile=str(path.name), tofile=str(path.name)))

patches = []
p = COMPAT / "model.py"
for old, new in [
    ("zero=torch.cuda.LongTensor(1).fill_(0)", "zero=torch.zeros(1, dtype=torch.long, device=source_ids.device)"),
    ("Beam(self.beam_size,self.sos_id,self.eos_id)", "Beam(self.beam_size,self.sos_id,self.eos_id,source_ids.device)"),
    ("def __init__(self, size,sos,eos):", "def __init__(self, size,sos,eos,device):"),
    ("self.tt = torch.cuda", "self.device = device"),
    ("self.scores = self.tt.FloatTensor(size).zero_()", "self.scores = torch.zeros(size, device=device)"),
    ("self.nextYs = [self.tt.LongTensor(size)\n                       .fill_(0)]", "self.nextYs = [torch.zeros(size, dtype=torch.long, device=device)]"),
    ("self.tt.LongTensor(self.nextYs[-1]).view(-1, 1)", "self.nextYs[-1].clone().view(-1, 1)"),
]:
    patches.append(replace_once(p, old, new))
patches.append(replace_once(COMPAT / "run.py", "WEIGHTS_NAME, AdamW,", "WEIGHTS_NAME,"))
patches.append(replace_once(COMPAT / "run.py", "import torch\n", "import torch\nfrom torch.optim import AdamW\n"))
patches.append(replace_once(COMPAT / "eval/bleu/bleu.py", "from c2nl.eval.bleu.bleu_scorer", "from .bleu_scorer"))
patches.append(replace_once(COMPAT / "eval/meteor/meteor.py",
    "['java', '-jar', '-Xmx{}'.format(mem), METEOR_JAR,",
    "['java', '-Xmx{}'.format(mem), '-jar', METEOR_JAR,"))
(RUN / "compatibility.patch").write_text("".join(patches))
sys.path.insert(0, str(COMPAT))
import run as upstream
from model import Seq2Seq
print("Source:", actual_commit)
print("Compatibility diff:", RUN / "compatibility.patch")

## 4. Unduh dataset resmi dan validasi

Manifest berikut memakai ID berkas Google Drive yang ditautkan repositori resmi.
Hanya `.code`, `.question`, dan `.answer` diperlukan. `.code.original` tidak dipakai baseline resmi.
Unduhan diperiksa terhadap ukuran publik; SHA-256 dicatat sebagai fingerprint lokal, bukan checksum resmi penulis.
Split asli dipertahankan, tanpa pembagian ulang atau deduplikasi.

In [ ]:
DATA_MANIFEST = {
  "python": {
    "train": [
      {
        "name": "train.code",
        "id": "14VtPI--blt6G40p5lv4yer9cXvI9O99P",
        "size": 16836207
      },
      {
        "name": "train.question",
        "id": "1OgEUW0DKx1wZicXxMuzw7kbvCvX93r_k",
        "size": 2279468
      },
      {
        "name": "train.answer",
        "id": "1dtWO_xDTX1Hhu19S1C3zZCjnXJtu-hmL",
        "size": 1310948
      }
    ],
    "dev": [
      {
        "name": "dev.answer",
        "id": "1-EKV-Kifi8KTQ2Iim1gdyLKqAMjhZSxD",
        "size": 211104
      },
      {
        "name": "dev.code",
        "id": "1kVy4qonwY-n8ePOmwVKcx9E6SVyqYH_u",
        "size": 2154031
      },
      {
        "name": "dev.question",
        "id": "1ofqNLaIldtYoO5w7elEwFZBvAqfUtx6n",
        "size": 273562
      }
    ],
    "test": [
      {
        "name": "test.answer",
        "id": "1b3muDqwFw1lwzBlaRuN1-6dKzXnZgyjE",
        "size": 210810
      },
      {
        "name": "test.code",
        "id": "1hb_KauZT24X6sNrVO0K0q8t6TdFnVYW6",
        "size": 2122036
      },
      {
        "name": "test.question",
        "id": "1PXuSe5H3LNUxiMMZ1YS9K02iOQGMkw9W",
        "size": 272866
      }
    ]
  },
  "java": {
    "train": [
      {
        "name": "train.answer",
        "id": "1zggaSuVvsfjtAr54gBRL9X445tgtv2yq",
        "size": 2546624
      },
      {
        "name": "train.code",
        "id": "1UUk_j-sqlj9ng558N5M79u2FuxsyxuYe",
        "size": 50769399
      },
      {
        "name": "train.question",
        "id": "1QSqGCggBK2FgAMxNg2NG8arUt5jhmTZ4",
        "size": 4561767
      }
    ],
    "dev": [
      {
        "name": "dev.code",
        "id": "1o7dzD3BU_HDQmlFu7fydHg47vbdrqiY7",
        "size": 6643693
      },
      {
        "name": "dev.question",
        "id": "1GP14w37AxZ901PLroD_BS3gef5iXdKfB",
        "size": 559560
      },
      {
        "name": "dev.answer",
        "id": "112oDzMQs83UOw90Go5k81T1NpcgnqwwL",
        "size": 423069
      }
    ],
    "test": [
      {
        "name": "test.answer",
        "id": "1g4BehMQ9fijNT-yU8UIqK5sRj90Pyx19",
        "size": 415398
      },
      {
        "name": "test.code",
        "id": "1WBqCYWvm4jn9Hyjdz3On3gVEakyyynXW",
        "size": 6503363
      },
      {
        "name": "test.question",
        "id": "15Ay0WO74bJzpLiVNw2hgAMT7eg0qrJnh",
        "size": 557825
      }
    ]
  }
}

In [ ]:
import gdown
EXPECTED = {"java": {"train": 95778, "dev": 12000, "test": 12000},
            "python": {"train": 56085, "dev": 7000, "test": 7000}}

def download_checked(file_id, path, size):
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists() or path.stat().st_size != size:
        partial = path.with_name(path.name + ".part")
        result = gdown.download(id=file_id, output=str(partial), quiet=False)
        if result is None or not partial.exists() or partial.stat().st_size != size:
            raise RuntimeError(f"Unduhan tidak lengkap: {path.name}. Periksa akses Google Drive atau kuota.")
        partial.replace(path)
    return path

fingerprints = {}
for split, entries in DATA_MANIFEST[LANGUAGE].items():
    for entry in entries:
        path = download_checked(entry["id"], DATA / split / entry["name"], entry["size"])
        fingerprints[f"{split}/{entry['name']}"] = hashlib.sha256(path.read_bytes()).hexdigest()
(RUN / "data_sha256.json").write_text(json.dumps(fingerprints, indent=2))

all_examples = {}
for split, expected in EXPECTED[LANGUAGE].items():
    counts = {suffix: len((DATA / split / f"{split}.{suffix}").read_text(encoding="utf-8").splitlines())
              for suffix in ("code", "question", "answer")}
    assert set(counts.values()) == {expected}, (split, counts, expected)
    rows = upstream.read_examples(str(DATA / split), split)
    assert len(rows) == expected
    assert all(row.source_code and row.source_question and row.target for row in rows), f"Baris kosong: {split}"
    all_examples[split] = rows
    print(split, counts)
examples = {split: rows[:LIMITS.get(split)] for split, rows in all_examples.items()}
print("Ukuran eksperimen:", {s: len(x) for s, x in examples.items()})
print("Contoh:", vars(examples["train"][0]))

## 5. Audit overlap kode

Penulis mengacak pasangan QA sebelum membagi dataset. Satu kode dapat muncul di beberapa split dengan pertanyaan berbeda.
Audit ini menghitung kesamaan **string kode**; ini bukan audit kemiripan AST atau pemisahan repository.
Jangan menghapus overlap saat membandingkan tabel paper karena benchmark akan berubah.

In [ ]:
train_codes = {row.source_code for row in all_examples["train"]}
overlap = {}
for split in ("dev", "test"):
    rows = all_examples[split]
    overlap[split] = {"qa_rows_with_code_in_train": sum(row.source_code in train_codes for row in rows),
                      "total_qa_rows": len(rows),
                      "unique_codes_shared_with_train": len({row.source_code for row in rows} & train_codes)}
print(json.dumps(overlap, indent=2))
(RUN / "code_overlap.json").write_text(json.dumps(overlap, indent=2))

## 6. Preprocessing resmi

Urutan input persis mengikuti implementasi: `<s> question </s> code </s>`.
Pertanyaan dan kode ditokenisasi terpisah, lalu digabung dan dipotong sesuai batas input.
Jangan menggantinya dengan tokenizer pasangan otomatis RoBERTa karena separator dapat berbeda.

In [ ]:
from types import SimpleNamespace
from transformers import RobertaConfig, RobertaTokenizer, RobertaModel, get_linear_schedule_with_warmup
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
tokenizer = RobertaTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
feature_args = SimpleNamespace(max_source_length=MAX_SOURCE_LENGTH, max_target_length=MAX_TARGET_LENGTH)

def tensor_data(rows, stage):
    features = upstream.convert_examples_to_features(rows, tokenizer, feature_args, stage=stage)
    names = ("source_ids", "source_mask", "target_ids", "target_mask")
    return TensorDataset(*(torch.tensor([getattr(f, name) for f in features], dtype=torch.long) for name in names))

train_data = tensor_data(examples["train"], "train") if MODE != "evaluate" else None
dev_rows = random.Random(SEED).sample(examples["dev"], min(1000, len(examples["dev"])))
dev_data = tensor_data(dev_rows, "test")
test_data = tensor_data(examples["test"], "test")
f = upstream.convert_examples_to_features(examples["train"][:1], tokenizer, feature_args, stage="dev")[0]
assert len(f.source_ids) == MAX_SOURCE_LENGTH
assert len(f.target_ids) == MAX_TARGET_LENGTH
assert f.source_ids[0] == tokenizer.cls_token_id
assert f.source_ids[sum(f.source_mask)-1] == tokenizer.sep_token_id
print("Tensor input:", dev_data.tensors[0].shape)
print("Awal token:", tokenizer.convert_ids_to_tokens(f.source_ids[:20]))

## 7. Evaluator resmi dan self-check

BLEU paper mengikuti nilai kedua `corpus_bleu` upstream: rata-rata BLEU-4 tiap contoh dengan smoothing.
Ini berbeda dari corpus BLEU, SacreBLEU, dan helper BLEU lain di repositori.
ROUGE-L memakai implementasi penulis; METEOR memakai JAR 1.5.
EM/F1 menormalisasi huruf kecil dan whitespace, **tanpa menghapus artikel atau tanda baca**.

In [ ]:
METRIC_NAMES = ["BLEU", "ROUGE-L", "METEOR", "EM", "F1"]

def score_answers(predictions, references):
    assert len(predictions) == len(references) > 0
    hypotheses = {i: [p] for i, p in enumerate(predictions)}
    gold = {i: [r] for i, r in enumerate(references)}
    _, bleu, _ = upstream.corpus_bleu(hypotheses, gold)
    rouge, _ = upstream.Rouge().compute_score(gold, hypotheses)
    meteor = upstream.Meteor()
    try:
        meteor_score, _ = meteor.compute_score(gold, hypotheses)
    finally:
        meteor.close()
    answer_scores = [upstream.compute_eval_score(p, [r]) for p, r in zip(predictions, references)]
    values = [bleu, rouge, meteor_score,
              np.mean([x[0] for x in answer_scores]), np.mean([x[3] for x in answer_scores])]
    return {name: float(value * 100) for name, value in zip(METRIC_NAMES, values)}

assert upstream.normalize_answer("  The CHILD.  ") == "the child."
assert upstream.compute_eval_score("Child", ["child"])[0] == 1
assert upstream.compute_eval_score("child.", ["child"])[0] == 0
assert upstream.compute_eval_score("a b", ["a c"])[3] == 0.5
perfect = score_answers(["the given child", "page arguments"], ["the given child", "page arguments"])
assert all(abs(perfect[name] - 100) < 1e-4 for name in ("BLEU", "ROUGE-L", "EM", "F1")), perfect
assert 0 < perfect["METEOR"] <= 100, perfect
print("Self-check evaluator lulus:", perfect)

## 8. Model CodeBERT + decoder Transformer

Model dan beam search berasal dari `model.py` resmi. Decoder baru diinisialisasi saat training.
Mode evaluate otomatis mengunduh checkpoint QA penulis dan memuatnya dengan strict state dict.
Untuk evaluate, encoder cukup dibuat dari config karena seluruh bobot ditimpa checkpoint QA.

In [ ]:
config = RobertaConfig.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
config._attn_implementation = "eager"
if MODE == "evaluate":
    encoder = RobertaModel(config)
else:
    encoder = RobertaModel.from_pretrained(MODEL_ID, revision=MODEL_REVISION,
                                          config=config, attn_implementation="eager")
decoder_layer = torch.nn.TransformerDecoderLayer(d_model=config.hidden_size, nhead=config.num_attention_heads)
decoder = torch.nn.TransformerDecoder(decoder_layer, num_layers=6)
model = Seq2Seq(encoder, decoder, config, beam_size=BEAM_SIZE, max_length=MAX_TARGET_LENGTH,
               sos_id=tokenizer.cls_token_id, eos_id=tokenizer.sep_token_id)
del encoder, decoder, decoder_layer

def load_weights(path):
    state = torch.load(path, map_location="cpu", weights_only=True)
    legacy_key = "encoder.embeddings.position_ids"
    if legacy_key in state and legacy_key not in model.state_dict():
        expected_positions = torch.arange(config.max_position_embeddings).unsqueeze(0)
        assert torch.equal(state[legacy_key], expected_positions), "Buffer posisi checkpoint tidak sesuai config."
        del state[legacy_key]
        print("Buffer posisi Transformers lama tervalidasi; bobot terlatih tetap dimuat secara strict.")
    model.load_state_dict(state, strict=True)

if MODE == "evaluate":
    if CHECKPOINT_PATH is None:
        checkpoint_id = {"java": "1RGwmoQ9xbr854RGvEcox_3lV89HCgF6N",
                         "python": "1ERr5zfCbyZ3JN4kGpSt5qu7rMkRAWqDl"}[LANGUAGE]
        CHECKPOINT_PATH = download_checked(checkpoint_id, WORK / "checkpoints" / f"{LANGUAGE}_codeqa.bin", 706916627)
    load_weights(CHECKPOINT_PATH)
model.to(DEVICE)
print("Parameter:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Encoder pretrained + decoder baru" if MODE != "evaluate" else f"Checkpoint QA: {CHECKPOINT_PATH}")

In [ ]:
@torch.no_grad()
def predict(dataset, label):
    model.eval()
    answers = []
    loader = DataLoader(dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False)
    for source_ids, source_mask, _, _ in tqdm(loader, desc=label):
        output = model(source_ids=source_ids.to(DEVICE), source_mask=source_mask.to(DEVICE))
        for beams in output:
            tokens = beams[0].cpu().tolist()
            if 0 in tokens:
                tokens = tokens[:tokens.index(0)]
            answers.append(tokenizer.decode(tokens, clean_up_tokenization_spaces=False))
    assert len(answers) == len(dataset)
    return answers

# Forward/backward kecil sebelum training panjang.
if MODE != "evaluate":
    model.train()
    batch = [t[:1].to(DEVICE) for t in train_data.tensors]
    loss, _, token_count = model(source_ids=batch[0], source_mask=batch[1],
                                  target_ids=batch[2], target_mask=batch[3])
    assert torch.isfinite(loss) and token_count > 0
    loss.backward()
    assert model.dense.weight.grad is not None
    model.zero_grad(set_to_none=True)
    print("Forward/backward lulus. Loss awal:", float(loss.detach()))
    del batch, loss, token_count
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

## 9. Fine-tuning dan pemilihan checkpoint

Full: 20 epoch, learning rate 5e-5, tanpa weight decay. Pilih checkpoint dari BLEU tertinggi pada
sampel dev tetap maksimum 1.000 contoh, mengikuti strategi kode resmi. Test tidak dipakai memilih checkpoint.
Gradient accumulation dibobot jumlah token target, termasuk sisa minibatch pada akhir epoch.

Checkpoint notebook menyimpan bobot model untuk inferensi, bukan state optimizer untuk resume training identik.
Gunakan direktori kerja persisten untuk training panjang. Jika VRAM habis pada microbatch 1,
gunakan GPU dengan memori lebih besar; jangan menganggap smoke sebagai hasil full.

In [ ]:
history = []
BEST_CHECKPOINT = RUN / "checkpoint-best-bleu.bin"
if MODE != "evaluate":
    loader = DataLoader(train_data, batch_size=MICRO_BATCH_SIZE, shuffle=True,
                        generator=torch.Generator().manual_seed(SEED))
    total_steps = math.ceil(len(loader) / ACCUMULATION) * EPOCHS
    no_decay = ("bias", "LayerNorm.weight")
    groups = [{"params": [p for n, p in model.named_parameters() if not any(x in n for x in no_decay)], "weight_decay": 0.0},
              {"params": [p for n, p in model.named_parameters() if any(x in n for x in no_decay)], "weight_decay": 0.0}]
    optimizer = torch.optim.AdamW(groups, lr=5e-5, eps=1e-8, foreach=False)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * 0.1), total_steps)
    best_bleu = -float("inf")
    for epoch in range(EPOCHS):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        loss_sum, tokens_sum, group_tokens = 0.0, 0, 0
        progress = tqdm(loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
        for step, batch in enumerate(progress):
            source_ids, source_mask, target_ids, target_mask = [t.to(DEVICE) for t in batch]
            loss, summed_loss, num_tokens = model(source_ids=source_ids, source_mask=source_mask,
                                                  target_ids=target_ids, target_mask=target_mask)
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Loss tidak finite pada epoch {epoch+1}, minibatch {step}")
            summed_loss.backward()
            group_tokens += int(num_tokens)
            loss_sum += float(summed_loss.detach())
            tokens_sum += int(num_tokens)
            if (step + 1) % ACCUMULATION == 0 or step + 1 == len(loader):
                for parameter in model.parameters():
                    if parameter.grad is not None:
                        parameter.grad.div_(group_tokens)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                group_tokens = 0
            progress.set_postfix(loss=round(loss_sum / tokens_sum, 4))
        predictions = predict(dev_data, "Dev checkpoint selection")
        metrics = score_answers(predictions, [row.target for row in dev_rows])
        record = {"epoch": epoch+1, "train_token_loss": loss_sum/tokens_sum, **metrics}
        history.append(record)
        (RUN / "history.json").write_text(json.dumps(history, indent=2))
        if metrics["BLEU"] > best_bleu:
            best_bleu = metrics["BLEU"]
            temporary = BEST_CHECKPOINT.with_suffix(".part")
            torch.save(model.state_dict(), temporary)
            temporary.replace(BEST_CHECKPOINT)
            (RUN / "best_dev.json").write_text(json.dumps(record, indent=2))
        print(record)
    del optimizer, scheduler, groups
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    load_weights(BEST_CHECKPOINT)
    print("Checkpoint terbaik dimuat:", BEST_CHECKPOINT)
else:
    print("Training dilewati: memakai checkpoint QA yang sudah dilatih penulis.")

## 10. Evaluasi akhir dan perbandingan paper

Tabel referensi di bawah adalah CodeBERT pada **seluruh test set**, bukan baris bertanda bintang yang hanya memakai 100 contoh.
Selisih pada mode smoke tidak bisa dipakai menilai reproduksi. Skor yang dihitung tidak diganti dengan angka paper.

In [ ]:
test_predictions = predict(test_data, "Test (best dev checkpoint)")
test_references = [row.target for row in examples["test"]]
test_metrics = score_answers(test_predictions, test_references)
PAPER_TEST = {"java": {"BLEU": 32.40, "ROUGE-L": 28.22, "METEOR": 10.10, "EM": 6.20, "F1": 29.20},
              "python": {"BLEU": 34.86, "ROUGE-L": 30.28, "METEOR": 12.51, "EM": 4.93, "F1": 31.56}}
from IPython.display import display, Markdown
lines = ["| Metrik | Hasil run | Paper test | Selisih |", "|---|---:|---:|---:|"]
for name in METRIC_NAMES:
    actual, target = test_metrics[name], PAPER_TEST[LANGUAGE][name]
    lines.append(f"| {name} | {actual:.2f} | {target:.2f} | {actual-target:+.2f} |")
display(Markdown("\n".join(lines)))
if MODE == "smoke":
    display(Markdown("**SMOKE ONLY: 8 train / 4 dev / 4 test, 1 epoch, beam 2. Bukan hasil reproduksi paper.**"))
with (RUN / "test_predictions.jsonl").open("w", encoding="utf-8") as out:
    for row, prediction in zip(examples["test"], test_predictions):
        out.write(json.dumps({"id": row.idx, "code": row.source_code, "question": row.source_question,
                              "reference": row.target, "prediction": prediction}, ensure_ascii=False) + "\n")
for row, prediction in list(zip(examples["test"], test_predictions))[:5]:
    print("Q:", row.source_question, "\nGold:", row.target, "\nPred:", prediction, "\n")

## 11. Simpan bukti eksperimen

Artefak memuat konfigurasi, versi paket, fingerprint dataset, audit overlap, history, prediksi, dan skor.
Full training serta evaluasi manusia tetap diperlukan untuk mengklaim reproduksi seluruh paper.
Model yang fasih belum tentu memahami kode; periksa kesalahan semantik pada prediksi.

In [ ]:
import importlib.metadata
settings = {"language": LANGUAGE, "mode": MODE, "source_commit": SOURCE_COMMIT,
            "model_id": MODEL_ID, "model_revision": MODEL_REVISION, "seed": SEED,
            "epochs": EPOCHS if MODE != "evaluate" else 0, "batch_reference": BATCH_REFERENCE,
            "effective_batch": EFFECTIVE_BATCH_SIZE, "micro_batch": MICRO_BATCH_SIZE,
            "gradient_accumulation": ACCUMULATION, "beam_size": BEAM_SIZE,
            "max_source_length": MAX_SOURCE_LENGTH, "max_target_length": MAX_TARGET_LENGTH,
            "learning_rate": 5e-5, "warmup_fraction": 0.1,
            "split_sizes": {s: len(x) for s, x in examples.items()},
            "dev_checkpoint_selection_size": len(dev_rows), "device": str(DEVICE),
            "python": platform.python_version(), "checkpoint_input": str(CHECKPOINT_PATH),
            "versions": {name: importlib.metadata.version(name) for name in
                         ("torch", "transformers", "gdown", "nltk", "numpy", "psutil")}}
(RUN / "config.json").write_text(json.dumps(settings, indent=2))
(RUN / "results.json").write_text(json.dumps({"mode": MODE, "language": LANGUAGE,
    "test_size": len(test_predictions), "metrics": test_metrics,
    "paper_test_reference": PAPER_TEST[LANGUAGE], "is_full_test": MODE != "smoke"}, indent=2))
(RUN / "environment.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))
print("Hasil disimpan:", RUN)
print("Mode:", MODE, "| Jumlah prediksi test:", len(test_predictions))

## Membaca hasil dan menjalankan ulang

- `smoke`: membuktikan alur data, forward/backward, training kecil, pemilihan checkpoint, dan metrik bekerja.
- `full`: ulangi dengan Python dan Java; gunakan `BATCH_REFERENCE="paper"` atau `"official_script"` secara eksplisit.
- `evaluate`: evaluasi checkpoint QA penulis di seluruh test split. Ini reproduksi evaluasi, bukan training ulang.
- Bandingkan hanya bahasa, split, beam, panjang input, serta metrik yang sama.
- Dataset diturunkan dari komentar dan pembagian dilakukan per pasangan QA; audit overlap penting untuk menafsirkan generalisasi.
- Seed sama tidak menjamin skor identik lintas versi PyTorch/CUDA, jenis GPU, atau pembagian minibatch.
- Notebook tidak membangun ulang dataset dari komentar, tidak melatih Seq2seq/Dual Encoder/Transformer terpisah,
  dan tidak mengulang penilaian manusia.

### Sitasi

```bibtex
@inproceedings{liu2021codeqa,
  title={CodeQA: A Question Answering Dataset for Source Code Comprehension},
  author={Liu, Chenxiao and Wan, Xiaojun},
  booktitle={Findings of the Association for Computational Linguistics: EMNLP 2021},
  pages={2618--2632},
  year={2021}
}
```